Connected to .venv (Python 3.12.8)

In [ ]:
{'promotion': 
        {
            'code': '299434',
            'name': 'AMBลดบันบัน',
            'coupon':'',
            'start_date': '04/01/2026',
            'end_date': '23/01/2026',
            'type':'Multi Condition',
            'trigger':{'value':'','types':''},
            'limit':{'transation':0,'day':0,'item':0},
            'reward':{'value': '19','types':'New Price'}, # type: ignore
            'note': '*บันบัน 1 ชิ้น แลกซื้อ 19 บาท ปกติ 30 บาท(สิทธิ์แลกซื้อท้ายใบเสร็จ และสิทธิ์แลกซื้อใน 7App ไม่สามารถใช้ร่วมกันได้)'
        },
        'entities': [
                {'bucket':1,
                    'item' :[
                            {
                            'code': '4105958',
                            'name': 'Hบันบันดั้งเดิม92ก',
                            'type':'Item',
                            'mode':'Include',
                            'barcode': {'value':'8851016300156','display': '*8851016300156*'},
                            'status': 'PENDING',
                            },
                        ],
                    },
                {'bucket':2,
                    'item' :[
    {
      'code': '4105958',
      'name': 'Hบันบันดั้งเดิม92ก',
      'type':'Item',
      'mode':'Include',
      'barcode': {'value':'8851016300156','display': '*8851016300156*'},
      'status': 'PENDING',
    },
  ],
  },
  ],
}

In [ ]:
{'report' :[{'file_name':'xxxxxxxxxxx','sheet':[{'sheet_name':'cccccccccccc','value':[{'row':0,'detail':{}}]}]}]}

In [ ]:
import openpyxl
from typing import Dict, List
import pandas as pd
storage_options = {
    'key': 'Administrator',
    'secret': 'Admin2000',
    'client_kwargs': {'endpoint_url': f'http://localhost:9000'}
}

class ExcelImportManager:
    @staticmethod
    def get_sheet_visibility(file_path: str) -> Dict[str, List[str]]:
        wb = openpyxl.load_workbook(file_path, read_only=True)
        
        visible_sheets = []
        hidden_sheets = []
        
        for sheet in wb.worksheets:
            if sheet.sheet_state == 'visible':
                visible_sheets.append(sheet.title)
            else:
                hidden_sheets.append(sheet.title)
                
        wb.close()
        
        return {
            'visible': visible_sheets,
            'hidden': hidden_sheets
        }

    @staticmethod
    def read_only_visible_sheets(file_path: str) -> Dict[str, pd.DataFrame]:
        sheet_info = ExcelImportManager.get_sheet_visibility(file_path)
        visible_sheet_names = sheet_info['visible']
        data_frames = pd.read_excel(file_path, storage_options=storage_options, sheet_name=visible_sheet_names, dtype=str)

        data_frames = pd.read_excel(file_path, sheet_name=visible_sheet_names)
        
        return data_frames

In [ ]:
import openpyxl
import pandas as pd
import io
from typing import Dict, List, Union
from urllib.parse import urlparse
import boto3
from botocore.client import Config

# ---------------------------------------------------------
# 1. ส่วนทดสอบการเชื่อมต่อ MinIO
# ---------------------------------------------------------
s3_test = boto3.client('s3', 
                  endpoint_url='http://localhost:9000', 
                  aws_access_key_id='Administrator', 
                  aws_secret_access_key='Admin2000',
                  config=Config(signature_version='s3v4'))

response = s3_test.list_buckets()
print('✅ เชื่อมต่อ MinIO สำเร็จ! นี่คือ Bucket ที่มีอยู่:')
for bucket in response['Buckets']:
    print(f' - {bucket['Name']}')

# ---------------------------------------------------------
# 2. คลาสจัดการ Excel
# ---------------------------------------------------------
class ExcelImportManager:
    @staticmethod
    def _get_file_stream(file_path: str) -> Union[str, io.BytesIO]:
        '''ฟังก์ชันดึงข้อมูลไฟล์รองรับ Local และ MinIO'''
        if file_path.startswith('s3://'):
            parsed_url = urlparse(file_path)
            bucket_name = parsed_url.netloc
            object_key = parsed_url.path.lstrip('/')
            
            # 📍 แก้ไข: ย้าย Config ของ MinIO เข้ามาไว้ในนี้ด้วย
            s3 = boto3.client('s3', 
                              endpoint_url='http://localhost:9000', 
                              aws_access_key_id='Administrator', 
                              aws_secret_access_key='Admin2000',
                              config=Config(signature_version='s3v4'))
                              
            obj = s3.get_object(Bucket=bucket_name, Key=object_key)
            file_stream = io.BytesIO(obj['Body'].read())
            file_stream.seek(0) # รีเซ็ต Cursor
            return file_stream
        
        return file_path

    @staticmethod
    def get_sheet_visibility(file_path: str) -> Dict[str, List[str]]:
        file_stream = ExcelImportManager._get_file_stream(file_path)
        
        wb = openpyxl.load_workbook(file_stream, data_only=True)
        
        visible_sheets = []
        hidden_sheets = []
        
        for sheet in wb.worksheets:
            if sheet.sheet_state == 'visible':
                visible_sheets.append(sheet.title)
            else:
                hidden_sheets.append(sheet.title)
                
        wb.close()
        return {'visible': visible_sheets, 'hidden': hidden_sheets}

    @staticmethod
    def read_only_visible_sheets(file_path: str) -> Dict[str, pd.DataFrame]:
        sheet_info = ExcelImportManager.get_sheet_visibility(file_path)
        visible_sheet_names = sheet_info['visible']
        
        file_stream = ExcelImportManager._get_file_stream(file_path)
        
        # ให้ Pandas อ่านเฉพาะ Sheet ที่กำหนด
        data_frames = pd.read_excel(file_stream, sheet_name=visible_sheet_names)
        
        return data_frames

# ---------------------------------------------------------
# 3. เรียกใช้งาน
# ---------------------------------------------------------
if __name__ == '__main__':
    file_path = 's3://promotion-files/20260307/00/ธันวาคม 2568 Corporate -ซื้อครบรับฟรีพรีเมียม - Pook.xlsx'
    
    print('\n⏳ กำลังดึงข้อมูลและแยก Sheet...')
    sheet_info = ExcelImportManager.get_sheet_visibility(file_path)
    print(f'✅ Sheet ที่แสดงอยู่: {sheet_info['visible']}')
    print(f'❌ Sheet ที่ซ่อนอยู่: {sheet_info['hidden']}')

    print('\n⏳ กำลังแปลงเป็น DataFrame...')
    data_frames = ExcelImportManager.read_only_visible_sheets(file_path)

    for sheet_name, df in data_frames.items():
        print(f'\n--- ข้อมูลจาก Sheet: {sheet_name} ---')
        print(df.head())

✅ เชื่อมต่อ MinIO สำเร็จ! นี่คือ Bucket ที่มีอยู่:
 - bucket-promotion
 - default-bucket
 - localhost
 - promotion
 - promotion-bucket
 - promotion-files

⏳ กำลังดึงข้อมูลและแยก Sheet...
✅ Sheet ที่แสดงอยู่: ['          Excel_Report         ']
❌ Sheet ที่ซ่อนอยู่: []

⏳ กำลังแปลงเป็น DataFrame...

--- ข้อมูลจาก Sheet:           Excel_Report          ---
   Promotion Code       Receipt Promotion Name  \
0          298268  ฟรีCalendarCardพี่จองคัลแลน   
1          298268  ฟรีCalendarCardพี่จองคัลแลน   
2          298268  ฟรีCalendarCardพี่จองคัลแลน   
3          298268  ฟรีCalendarCardพี่จองคัลแลน   
4          298268  ฟรีCalendarCardพี่จองคัลแลน   

                   Promotion Name   Promotion Type    Group Name Active From  \
0  ครบ150บ.ขึ้นไป ฟรีCalendarCard  Multi Condition  ฟรีพรีเมี่ยม  15/12/2025   
1  ครบ150บ.ขึ้นไป ฟรีCalendarCard  Multi Condition  ฟรีพรีเมี่ยม  15/12/2025   
2  ครบ150บ.ขึ้นไป ฟรีCalendarCard  Multi Condition  ฟรีพรีเมี่ยม  15/12/2025   
3  ครบ150บ.ขึ้นไป ฟรีCa

In [ ]:
import os 
os.environ['MINIO_ENDPOINT'] = 'http://localhost:9000'
os.environ['MINIO_ACCESS_KEY'] = 'Administrator' # ตัวอย่าง Access Key
os.environ['MINIO_SECRET_KEY'] = 'Admin2000' # ตัวอย่าง Secret Key
file_path = 's3://promotion-files/20260307/00/ธันวาคม 2568 Corporate -ซื้อครบรับฟรีพรีเมียม - Pook.xlsx'

# โค้ดที่เหลือเรียกเหมือนเดิมเป๊ะครับ
data_frames = ExcelImportManager.read_only_visible_sheets(file_path)
for sheet_name, df in data_frames.items():
    print(df.head())

ClientError: An error occurred (InvalidAccessKeyId) when calling the GetObject operation: The AWS Access Key Id you provided does not exist in our records.

In [ ]:
# สมมติว่าไฟล์ ExcelImportManager อยู่ในไฟล์เดียวกับสคริปต์นี้
# หรือ import เข้ามา: from app.backend.utils.excel_manager import ExcelImportManager

file_path = r'D:\gosoft_qc\พฤศจิกายน 2568 7Delivery-Pook.xlsx'
file_path = f's3://promotion-files/20260307/00/ธันวาคม 2568 Corporate -ซื้อครบรับฟรีพรีเมียม - Pook.xlsx'

# 1. ดึงแค่ชื่อ Sheet มาดูก่อน (แยก ซ่อน/ไม่ซ่อน)
sheet_info = ExcelImportManager.get_sheet_visibility(file_path)
print(f'✅ Sheet ที่แสดงอยู่: {sheet_info['visible']}')
print(f'❌ Sheet ที่ซ่อนอยู่: {sheet_info['hidden']}')

# 2. สั่งอ่านข้อมูล (DataFrame) เฉพาะ Sheet ที่แสดงอยู่
# จะคืนค่ากลับมาเป็น Dictionary ที่มี Key เป็นชื่อ Sheet และ Value เป็นข้อมูล
data_frames = ExcelImportManager.read_only_visible_sheets(file_path)

for sheet_name, df in data_frames.items():
    print(f'\n--- ข้อมูลจาก Sheet: {sheet_name} ---')
    print(df.head()) # ปริ้นท์ดูข้อมูล 5 บรรทัดแรก

ClientError: An error occurred (InvalidAccessKeyId) when calling the GetObject operation: The AWS Access Key Id you provided does not exist in our records.

In [ ]:
import ast
import logging
from pathlib import Path
from typing import Set, Tuple


# ---------------- LOGGING ----------------

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)

logger = logging.getLogger('Scanner')


# ---------------- AST VISITOR ----------------

class FunctionCallVisitor(ast.NodeVisitor):

    def __init__(self, file_path: Path):
        self.file_path = str(file_path)
        self.current_function = None
        self.calls: Set[Tuple[str, str]] = set()

    def visit_FunctionDef(self, node: ast.FunctionDef):
        prev = self.current_function
        self.current_function = f'{self.file_path}:{node.name}'

        self.generic_visit(node)

        self.current_function = prev

    def visit_Call(self, node: ast.Call):
        if self.current_function:
            name = self._get_name(node.func)
            if name:
                self.calls.add((self.current_function, name))

        self.generic_visit(node)

    def _get_name(self, node):
        if isinstance(node, ast.Name):
            return node.id
        if isinstance(node, ast.Attribute):
            return node.attr
        return None


# ---------------- MAIN SCANNER ----------------

class GlobalFunctionScanner:

    IGNORE_DIRS = {
        '.git',
        '__pycache__',
        'venv',
        '.venv',
        'node_modules',
        'dist',
        'build'
    }

    def __init__(self, root: str, dry_run=False):
        self.root = Path(root)
        self.graph: Set[Tuple[str, str]] = set()
        self.dry_run = dry_run

    # ---------------------------------

    def scan_all_python_files(self):

        logger.info('Scanning folders...')

        for path in self.root.rglob('*.py'):

            if any(part in self.IGNORE_DIRS for part in path.parts):
                continue

            yield path

    # ---------------------------------

    def parse_file(self, file: Path):

        try:
            source = file.read_text(encoding='utf-8')
            tree = ast.parse(source)
        except PermissionError:
            logger.warning(f'Permission denied: {file}')
            return
        except Exception as e:
            logger.warning(f'Parse fail {file}: {e}')
            return

        visitor = FunctionCallVisitor(file)
        visitor.visit(tree)

        self.graph.update(visitor.calls)

    # ---------------------------------

    def build_graph(self):

        for file in self.scan_all_python_files():
            logger.info(f'Parsing {file}')
            self.parse_file(file)

    # ---------------------------------

    def export_mermaid(self, output='GLOBAL_FUNCTION_FLOW.md'):

        if self.dry_run:
            logger.info('Dry-run enabled (skip export)')
            return

        lines = ['```mermaid', 'graph TD']

        for caller, callee in sorted(self.graph):
            caller = caller.replace(''', '').replace('.', '_')
            callee = callee.replace(''', '').replace('.', '_')

            lines.append(f'    '{caller}' --> '{callee}'')

        lines.append('```')

        Path(output).write_text('\n'.join(lines), encoding='utf-8')

        logger.info(f'Diagram created: {output}')

    # ---------------------------------

    def run(self):

        logger.info(f'Start scanning root: {self.root}')

        self.build_graph()
        self.export_mermaid()

        logger.info(f'Total edges: {len(self.graph)}')

In [ ]:
import ast
from pathlib import Path
from typing import Dict, List, Set


class FunctionCallVisitor(ast.NodeVisitor):
    '''
    AST Visitor สำหรับเก็บ function call
    '''

    def __init__(self, file_name: str):
        self.file_name = file_name
        self.current_function = None
        self.calls = []

    def visit_FunctionDef(self, node: ast.FunctionDef):
        prev = self.current_function
        self.current_function = f'{self.file_name}:{node.name}'

        self.generic_visit(node)

        self.current_function = prev

    def visit_Call(self, node: ast.Call):
        if self.current_function:
            func_name = self._get_call_name(node.func)
            if func_name:
                self.calls.append(
                    (self.current_function, func_name)
                )

        self.generic_visit(node)

    def _get_call_name(self, node):
        if isinstance(node, ast.Name):
            return node.id

        if isinstance(node, ast.Attribute):
            return node.attr

        return None


# --------------------------------------------------


class ProjectScanner:
    '''
    Scan python project และสร้าง function flow diagram
    '''

    def __init__(self, root_path: str):
        self.root = Path(root_path)
        self.graph: Set[tuple] = set()

    # --------------------------

    def scan_files(self) -> List[Path]:
        return list(self.root.rglob('*.py'))

    # --------------------------

    def parse_file(self, file: Path):

        try:
            source = file.read_text(encoding='utf-8')
            tree = ast.parse(source)
        except Exception:
            return

        visitor = FunctionCallVisitor(file.name)
        visitor.visit(tree)

        for edge in visitor.calls:
            self.graph.add(edge)

    # --------------------------

    def build_call_graph(self):

        for file in self.scan_files():
            self.parse_file(file)

    # --------------------------

    def export_mermaid(self, output='function_flow.md'):

        lines = ['```mermaid', 'graph TD']

        for caller, callee in sorted(self.graph):
            caller = caller.replace('.', '_')
            callee = callee.replace('.', '_')
            lines.append(f'    '{caller}' --> '{callee}'')

        lines.append('```')

        Path(output).write_text('\n'.join(lines), encoding='utf-8')

        print(f'Diagram generated -> {output}')

    # --------------------------

    def run(self):
        self.build_call_graph()
        self.export_mermaid()

In [ ]:
scanner = GlobalFunctionScanner('./app/')
scanner.run()

In [ ]:
import sqlite3
from pathlib import Path
from datetime import datetime
import os


class FilePathIndexer:

    def __init__(self, root_path: str, db_path: str = 'files.db'):
        self.root_path = Path(root_path).resolve()
        self.db_path = db_path

        if not self.root_path.exists():
            raise FileNotFoundError(self.root_path)

        self.conn = sqlite3.connect(self.db_path)
        self._create_table()

    def _create_table(self):
        self.conn.execute('''
        CREATE TABLE IF NOT EXISTS files (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            file_name TEXT,
            full_path TEXT UNIQUE,
            extension TEXT,
            size INTEGER,
            created_at TEXT,
            modified_at TEXT
        )
        ''')
        self.conn.commit()

    def scan(self):
        cursor = self.conn.cursor()

        for root, _, files in os.walk(self.root_path):
            for name in files:
                file_path = Path(root) / name

                try:
                    stat = file_path.stat()

                    cursor.execute('''
                        INSERT OR IGNORE INTO files
                        (file_name, full_path, extension, size,
                         created_at, modified_at)
                        VALUES (?, ?, ?, ?, ?, ?)
                    ''', (
                        name,
                        str(file_path),
                        file_path.suffix.lower(),
                        stat.st_size,
                        datetime.fromtimestamp(stat.st_ctime).isoformat(),
                        datetime.fromtimestamp(stat.st_mtime).isoformat(),
                    ))

                except PermissionError:
                    print('Permission denied:', file_path)

        self.conn.commit()

    def close(self):
        self.conn.close()




In [ ]:
# ===== RUN =====
if __name__ == '__main__':
    indexer = FilePathIndexer(
        root_path=r'C:\Users\nakarinsue\Documents', 
        db_path='files.db'
    )

    indexer.scan()
    indexer.close()

    print('Index completed.')

In [ ]:
import requests
import json

url = 'https://sdl-master-api-uat.cpall.co.th/api/products/internal'

payload = json.dumps({
 'storeId': '11104',
 'productCode': [
  '5100320','5100322','5100323'
 ],
 'checkInventory': True,
 'checkPeriodTime': True,
 'ContentLanguage': 'th'
})
headers = {
	'x-api-key': '4K2kmEfGTYZHVXFNqdKTAX3MES5NYvWV',
	'device_type': '3',
	'api-version': '7',
	'Content-Type': 'application/json'
}

response = requests.request('POST', url, headers=headers, data=payload)

print(response.text)


{"returnCode":"0000","returnMessage":"success","result":[{"product_barcode":"8858746100926","product_barcode_list":["8858746100926"],"product_code":"5100320","product_name":"ถุงขยะ my items 24x28นิ้ว","product_image":"https://media-uat.cpall.co.th/products/5100320_050920221403.jpg","product_image_hd":"https://media-uat.cpall.co.th/products/5100320_050920221403.jpg","product_image_list":["https://media-uat.cpall.co.th/products/5100320_050920221403.jpg"],"product_sell_price":39,"product_hq_price":39,"product_type_id":3,"product_type_name":"on shelf","isSaleWithOther":true,"product_description":"<p>ถุงขยะ my items เหนียว ทนทาน ปราศจากกลิ่น รับน้ำหนักได้มาก ออกแบบมาให้หยิบใช้งานง่าย</p>","product_type_order":3,"pma":"51","cat_pma":"01","sub_cat_pma":"04","is_one_touch_item":false,"product_sections_total_required_qty":0,"is_product_sections_required":false,"product_sections":[],"product_discount_price":0,"is_dynamic_price":false,"product_min_price":0,"product_max_price":0,"promotions":[{"pr

In [ ]:
[
  {
    'header': {
      'pro_code': 288455,
      'end_date': '2025-11-23',
      'limit_day': 0,
      'pro_name': 'ส่วนลดศรีจันทร์เอ็นชานเท็ดสีเนื้อ ด.',
      'limit_item': 0,
      'pro_receipt_name': 'ส่วนลดศรีจันทร์เอ็นชานเท็ดสีเนื้อ ด.',
      'limit_redemp': 0,
      'reward_value': '39.00',
      'pro_group': 'ตลาดนัด',
      'reward_type': 'New Price',
      'reward_ma': '',
      'member_requ': '',
      'reward_name': '',
      'notes': 'รองพื้นศรีจันทร์เอ็นชานเท็ด #120สีเนื้อ ด. 1 ชิ้น พิเศษ 39 บาท ปกติ 49 บาท ( เฉพาะสาขาที่ร่วมรายการ )',
      'start_date': '2025-10-24',
      'limit_tran': 10,
    },
    'file': {
      'file_name': '00213-ครั้งที่ 1 โปรโมชั่น เดือนพฤศจิกายน 2568 ตลาดนัด 00213 ( Test ).xlsx',
      'user_mk': '',
      'sheet': 'ตลาดนัด',
    },
    'bucket': {
      'bucket': 1,
      'trigger_value': '1',
      'trigger_type': 'Quantity',
      'barcode': '8858696808200',
      'entity_name': 'pรองพื้นศรีจันทร์เอ็นชานเท็ด 120สีเนื้อ(ด) 7 ก.',
      'coupon': '',
      'entity_code': '5003106',
      'entity_type': 'Item',
      'condition': '',
      'mode': 'Include',
      'condition_name': 'รองพื้นศรีจันทร์เอ็นชานเท็ด #120 ด.',
      'condition_id': ''
    }
  }
]

TypeError: str.format_map() takes exactly one argument (0 given)

In [ ]:
from minio import Minio
from minio.error import S3Error
import io

# การตั้งค่า Client (ควรดึงค่าจาก Environment Variables)
minio_client = Minio(
    'localhost:9000',
    access_key='Administrator',
    secret_key='Admin2000',
    secure=False # เปลี่ยนเป็น True หากใช้ HTTPS
)

BUCKET_NAME = 'promotion-files'

# --- 1. ฟังก์ชันดึงไฟล์ (Get File) ---
def get_minio_file(file_name: str):
    '''ดึงข้อมูลไฟล์จาก MinIO ในรูปแบบ Bytes'''
    try:
        response = minio_client.get_object(BUCKET_NAME, file_name)
        return response.read()
    except S3Error as e:
        print(f'Error getting file: {e}')
        return None
    finally:
        if 'response' in locals():
            response.close()
            response.release_conn()

# --- 2. ฟังก์ชันลบไฟล์ (Delete File) ---
def delete_minio_file(file_name: str):
    '''ลบไฟล์ออกจาก Bucket'''
    try:
        minio_client.remove_object(BUCKET_NAME, file_name)
        return True
    except S3Error as e:
        print(f'Error deleting file: {e}')
        return False

# --- 3. ฟังก์ชันแก้ไขชื่อไฟล์ (Rename/Move) ---
def rename_minio_file(old_name: str, new_name: str):
    '''
    แก้ไขชื่อไฟล์ใน MinIO (ใช้วิธี Copy ไปชื่อใหม่แล้วลบชื่อเดิม)
    '''
    try:
        # 1. Copy ไฟล์ไปยังชื่อใหม่
        from minio.commonconfig import CopySource
        minio_client.copy_object(
            BUCKET_NAME,
            new_name,
            CopySource(BUCKET_NAME, old_name),
        )
        # 2. ลบไฟล์เดิมออก
        minio_client.remove_object(BUCKET_NAME, old_name)
        return True
    except S3Error as e:
        print(f'Error renaming file: {e}')
        return False
def delete_minio_bucket(bucket_name: str, force: bool = False):
    '''
    ลบ Bucket ออกจากระบบ
    :param bucket_name: ชื่อ Bucket ที่ต้องการลบ
    :param force: หากเป็น True จะลบไฟล์ทั้งหมดข้างในก่อนลบ Bucket
    '''
    try:
        # ตรวจสอบว่ามี Bucket อยู่จริงหรือไม่
        if not minio_client.bucket_exists(bucket_name):
            print(f'Bucket '{bucket_name}' does not exist.')
            return False

        if force:
            # ลบไฟล์ทั้งหมดใน Bucket ก่อน (มาตรฐาน S3 ต้องว่างก่อนลบ)
            objects_to_delete = minio_client.list_objects(bucket_name, recursive=True)
            for obj in objects_to_delete:
                minio_client.remove_object(bucket_name, obj.object_name)
            print(f'All objects in '{bucket_name}' have been deleted.')

        # ลบ Bucket
        minio_client.remove_bucket(bucket_name)
        print(f'Bucket '{bucket_name}' deleted successfully.')
        return True

    except S3Error as e:
        print(f'Error occurred: {e}')
        return False

In [38]:
import random
import base64
import requests
import xmltodict as xd
import pandas as pd
from datetime import datetime, timedelta
import oracledb
from dataclasses import dataclass, field
from typing import Any, Dict, Optional,Literal
import json
# ตั้งค่า Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# ==========================================
# 1. Data Models (โครงสร้างข้อมูล)
# ==========================================

@dataclass
class StoreConfig:
    store_id: str = "09892"
    zone: str = "1"
    employee_id: str = "0555505"
    pos_tax_id: str = "1537264827382"
    vendor_id: str = "82204"
    service_id: str = "00"
    item_name: str = "Test"
    vat_amt: str = "0"
    rept_type: str = "H"
    payment_channel: str = "C05"

@dataclass
class CustomerInfo:
    name: str = ""
    addr_1: str = ""
    addr_2: str = ""
    addr_3: str = ""
    phone_no: str = ""

@dataclass
class BillInfo:
    amt_min: str = "1"
    amt_max: str = "90000"
    bill_amt: str = "50"

@dataclass
class TransactionContext:
    """Class เก็บ Context ของการทำ Transaction ทั้งหมด แทนการใช้ Array ซ้อน Array"""
    store: StoreConfig = field(default_factory=StoreConfig)
    customer: CustomerInfo = field(default_factory=CustomerInfo)
    bill: BillInfo = field(default_factory=BillInfo)
    
    # ข้อมูล Data 1-9 (รองรับ str, int, list)
    data_1: Any = None
    data_2: Any = None
    data_3: Any = None
    data_4: Any = None
    data_5: Any = None
    data_6: Any = None
    data_7: Any = None
    data_9: Any = None
    
    step: int = 1
    bus_date: str = field(default_factory=lambda: datetime.now().strftime("%Y/%m/%d"))
    bus_time: str = field(default_factory=lambda: datetime.now().strftime("%X"))
    common_trn_id: str = field(default_factory=lambda: str(random.randint(0, 100)))

    # ข้อมูลที่ดึงกลับมาจาก Response ก่อนหน้า (ใช้ทำ Cancel / Confirm)
    ref_data: Dict[str, str] = field(default_factory=dict)

    def prepare_data(self):
        """แปลงค่า int (Gen เลขสุ่ม) และ list (วนลูปตาม step) ให้เป็น String พร้อมใช้งาน"""
        data_fields = ['data_1', 'data_2', 'data_3', 'data_4', 'data_5', 'data_6', 'data_7', 'data_9']
        for attr in data_fields:
            val = getattr(self, attr)
            if isinstance(val, int):
                setattr(self, attr, ''.join([str(random.randint(0, 9)) for _ in range(val)]))
            elif isinstance(val, list):
                setattr(self, attr, str(val[self.step % len(val)]))
            elif val is None:
                setattr(self, attr, "")
            else:
                setattr(self, attr, str(val))
        self.step += 1

    def load_reference_from_response(self, response_dict: Dict[str, Any]):
        """
        ดึงและคำนวณค่าจาก Response ก่อนหน้า เพื่อใช้ประกอบ Payload ของ Action ถัดไป
        ครอบคลุม Logic การทำ Substring และการบวกค่า SUM_SEQ, SUM_AMT จากโค้ดต้นฉบับ
        """
        tx_id_full = str(response_dict.get('TX_ID', ''))
        bill_amt_full = str(response_dict.get('BILL_AMT', '0'))
        
        # 1. จัดการ SEQ_NO และ TX_ID (อ้างอิง: ตัด "145" ออกตาม logic เดิม)
        seq_no1 = tx_id_full.replace("145", "")
        tx_id2 = tx_id_full[:8] if len(tx_id_full) >= 8 else tx_id_full
        
        sum_seq = ""
        sum_amt = ""
        
        # 2. คำนวณ SUM_SEQ และ SUM_AMT
        # ใช้ .zfill(5) แทนฟังก์ชัน CHECK_Length เดิม เพื่อเติมเลข 0 ด้านหน้าให้ครบ 5 หลัก
        if len(seq_no1) == 5:
            # กรณี Single Transaction
            try:
                sum_seq = str(int(seq_no1) + 1).zfill(5)
                sum_amt = str(float(bill_amt_full) + 2)
            except ValueError:
                sum_seq = seq_no1
                sum_amt = bill_amt_full
                
        elif len(seq_no1) > 5:
            # กรณี Multiple Transaction (มีการใช้เครื่องหมาย | คั่น)
            try:
                # โค้ดเดิม: ดึงตำแหน่ง [0:5] และ [6:11] มาบวก 2
                seq_no2 = str(int(seq_no1[0:5]) + 2).zfill(5)
                seq_no3 = str(int(seq_no1[6:11]) + 2).zfill(5) if len(seq_no1) >= 11 else ""
                sum_seq = f"{seq_no2}|{seq_no3}" if seq_no3 else seq_no2
                
                # จัดการ BILL_AMT ที่มี | คั่น
                if "|" in bill_amt_full:
                    parts = bill_amt_full.split("|")
                    amt1 = str(float(parts[0]) + 2)
                    amt2 = parts[1] 
                    sum_amt = f"{amt1}|{amt2}"
                else:
                    sum_amt = str(float(bill_amt_full) + 2)
            except Exception:
                # ป้องกัน Error กรณี Parsing ไม่สำเร็จ ให้ใช้ค่าตั้งต้น
                sum_seq = seq_no1
                sum_amt = bill_amt_full

        # 3. จัดเก็บลง Dictionary เพื่อให้ SoapPayloadBuilder เรียกใช้ได้ง่าย
        self.ref_data = {
            'VENDOR_ID': str(response_dict.get('VENDOR_ID', '')),
            'SERV_ID': str(response_dict.get('SERV_ID', '')),
            'TX_ID': tx_id2,
            'SEQ_NO': seq_no1,
            'SUM_SEQ': sum_seq,      # ค่าที่ถูกบวก +1 หรือ +2 แล้ว
            'BILL_AMT': bill_amt_full,
            'SUM_AMT': sum_amt,      # ค่าที่ถูกบวก +2 แล้ว
            
            # เก็บข้อมูล DATA 1-9
            'DATA_1': str(response_dict.get('DATA_1', '')),
            'DATA_2': str(response_dict.get('DATA_2', '')),
            'DATA_3': str(response_dict.get('DATA_3', '')),
            'DATA_4': str(response_dict.get('DATA_4', '')),
            'DATA_5': str(response_dict.get('DATA_5', '')),
            'DATA_6': str(response_dict.get('DATA_6', '')),
            'DATA_7': str(response_dict.get('DATA_7', '')),
            'DATA_9': str(response_dict.get('DATA_9', '')),
            
            # เก็บข้อมูลลูกค้า (แมปจาก XML Response ที่คุณตั้งไว้ใน Llistout)
            'CUST_NAME': str(response_dict.get('CUSTOMER_NAME', '')),
            'CUST_ADDR_1': str(response_dict.get('CUSTOMER_ADDR_1', '')),
            'CUST_ADDR_2': str(response_dict.get('CUSTOMER_ADDR_2', '')),
            'CUST_ADDR_3': str(response_dict.get('CUSTOMER_ADDR_3', '')),
            'CUST_PHONE_NO': str(response_dict.get('CUSTOMER_TEL_NO', ''))
        }

# ==========================================
# 2. Payload Builders (สร้าง XML)
# ==========================================

class SoapPayloadBuilder:
    """ทำหน้าที่สร้าง XML สำหรับยิง API ทุกรูปแบบ (แยกส่วนหน้าที่ชัดเจน ตามมาตรฐาน Enterprise)"""
    
    @staticmethod
    def build_data_exchange(tx: TransactionContext) -> str:
        return f"""<?xml version="1.0" encoding="UTF-8"?><HQ_REQUEST><SERVICE_BOX><ADDRESS><VENDOR_ID>{tx.store.vendor_id}</VENDOR_ID><SERVICE_ID>{tx.store.service_id}</SERVICE_ID><METHOD>DataExchange</METHOD></ADDRESS><DATA><PAYMENT_CHANNEL>{tx.store.payment_channel}</PAYMENT_CHANNEL><VENDOR_ID>{tx.store.vendor_id}</VENDOR_ID><SERV_ID>{tx.store.service_id}</SERV_ID><SERVICE_ID>{tx.store.service_id}</SERVICE_ID><STORE_ID>{tx.store.store_id}</STORE_ID><STATION_ID>1</STATION_ID><BUS_DATE>{tx.bus_date}</BUS_DATE><BUS_TIME>{tx.bus_time}</BUS_TIME><SYS_DATE>{tx.bus_date}</SYS_DATE><SYS_TIME>{tx.bus_time}</SYS_TIME><COMMON_TRN_ID>{tx.common_trn_id}</COMMON_TRN_ID><SEQ_NO></SEQ_NO><CLIENT_SERV_SEQ></CLIENT_SERV_SEQ><SHIFT_ID>9</SHIFT_ID><TRANS_TYPE>N</TRANS_TYPE><ACCT_NO></ACCT_NO><BILL_AMT>{tx.bill.bill_amt}</BILL_AMT><ROUND_BILL_AMT>{tx.bill.bill_amt}</ROUND_BILL_AMT><VAT_AMT>{tx.store.vat_amt}</VAT_AMT><REPT_TYPE>{tx.store.rept_type}</REPT_TYPE><REPT_NO></REPT_NO><PREV_REF_SEQ></PREV_REF_SEQ><PREV_REF_DATE></PREV_REF_DATE><SERV_CHARGE_NO></SERV_CHARGE_NO><ITEM_NAME>{tx.store.item_name}</ITEM_NAME><ITEM_SELECTION>N</ITEM_SELECTION><EMPLOYEE_ID>{tx.store.employee_id}</EMPLOYEE_ID><POS_TAX_ID>{tx.store.pos_tax_id}</POS_TAX_ID><DATA_1>{tx.data_1}</DATA_1><DATA_2>{tx.data_2}</DATA_2><DATA_3>{tx.data_3}</DATA_3><DATA_4>{tx.data_4}</DATA_4><DATA_5>{tx.data_5}</DATA_5><DATA_6>{tx.data_6}</DATA_6><DATA_7>{tx.data_7}</DATA_7><DATA_9>{tx.data_9}</DATA_9><ZONE>{tx.store.zone}</ZONE><PAYMENT_TYPE>001</PAYMENT_TYPE><CANCEL_ID></CANCEL_ID><CUST_NAME>{tx.customer.name}</CUST_NAME><CUST_ADDR_1>{tx.customer.addr_1}</CUST_ADDR_1><CUST_ADDR_2>{tx.customer.addr_2}</CUST_ADDR_2><CUST_ADDR_3>{tx.customer.addr_3}</CUST_ADDR_3><CUST_PHONE_NO>{tx.customer.phone_no}</CUST_PHONE_NO></DATA></SERVICE_BOX></HQ_REQUEST>"""

    @staticmethod
    def build_cancel(tx: TransactionContext) -> str:
        ref = tx.ref_data
        return f"""<?xml version="1.0" encoding="UTF-8"?><HQ_REQUEST><SERVICE_BOX><ADDRESS><VENDOR_ID>{tx.store.vendor_id}</VENDOR_ID><SERVICE_ID>{tx.store.service_id}</SERVICE_ID><METHOD>Cancel</METHOD></ADDRESS><DATA><PAYMENT_CHANNEL>{tx.store.payment_channel}</PAYMENT_CHANNEL><VENDOR_ID>{ref.get('VENDOR_ID', '')}</VENDOR_ID><SERV_ID>{ref.get('SERV_ID', '')}</SERV_ID><SERVICE_ID>{ref.get('SERV_ID', '')}</SERVICE_ID><STORE_ID>{tx.store.store_id}</STORE_ID><STATION_ID>1</STATION_ID><BUS_DATE>{tx.bus_date}</BUS_DATE><BUS_TIME>{tx.bus_time}</BUS_TIME><TX_ID>{ref.get('TX_ID', '')}</TX_ID><PAYMENT_TYPE>001</PAYMENT_TYPE><CANCEL_ID></CANCEL_ID></DATA></SERVICE_BOX></HQ_REQUEST>"""

    @staticmethod
    def build_data_exchange_confirm(tx: TransactionContext) -> str:
        ref = tx.ref_data
        return f"""<?xml version="1.0" encoding="UTF-8"?><HQ_REQUEST><SERVICE_BOX><ADDRESS><VENDOR_ID>{tx.store.vendor_id}</VENDOR_ID><SERVICE_ID>{tx.store.service_id}</SERVICE_ID><METHOD>DataExchangeConfirm</METHOD></ADDRESS><DATA><PAYMENT_CHANNEL>{tx.store.payment_channel}</PAYMENT_CHANNEL><VENDOR_ID>{ref.get('VENDOR_ID', '')}</VENDOR_ID><SERV_ID>{ref.get('SERV_ID', '')}</SERV_ID><SERVICE_ID>{ref.get('SERV_ID', '')}</SERVICE_ID><STATION_ID>1</STATION_ID><STORE_ID>{tx.store.store_id}</STORE_ID><BUS_DATE>{tx.bus_date}</BUS_DATE><BUS_TIME>{tx.bus_time}</BUS_TIME><SYS_DATE>{tx.bus_date}</SYS_DATE><SYS_TIME>{tx.bus_time}</SYS_TIME><TX_ID>{ref.get('TX_ID', '')}</TX_ID><SEQ_NO>{ref.get('SEQ_NO', '')}</SEQ_NO><EMPLOYEE_ID>{tx.store.employee_id}</EMPLOYEE_ID><CLIENT_SERV_SEQ>{ref.get('SEQ_NO', '')}</CLIENT_SERV_SEQ><SERV_ID>{tx.store.service_id}</SERV_ID><BILL_AMT>{ref.get('BILL_AMT', '')}</BILL_AMT><ROUND_BILL_AMT>{ref.get('BILL_AMT', '')}</ROUND_BILL_AMT><ACCT_NO></ACCT_NO><VAT_AMT>{tx.store.vat_amt}</VAT_AMT><DATA_1>{ref.get('DATA_1', '')}</DATA_1><DATA_2>{ref.get('DATA_2', '')}</DATA_2><DATA_3>{ref.get('DATA_3', '')}</DATA_3><DATA_4>{ref.get('DATA_4', '')}</DATA_4><DATA_5>{ref.get('DATA_5', '')}</DATA_5><DATA_6>{ref.get('DATA_6', '')}</DATA_6><DATA_7>{ref.get('DATA_7', '')}</DATA_7><DATA_9>{ref.get('DATA_9', '')}</DATA_9><ZONE>{tx.store.zone}</ZONE><PAYMENT_TYPE>001</PAYMENT_TYPE><TOT_BILL_TRANS></TOT_BILL_TRANS><TOT_BILL_AMT></TOT_BILL_AMT><TOT_VENDOR_TRANS></TOT_VENDOR_TRANS><TOT_VENDOR_AMT></TOT_VENDOR_AMT><TOT_COUNTER_TRANS></TOT_COUNTER_TRANS><TOT_COUNTER_AMT></TOT_COUNTER_AMT><TOT_CLIENT_TRANS></TOT_CLIENT_TRANS><TOT_CLIENT_AMT></TOT_CLIENT_AMT><TOT_BILL_TRANS_OR></TOT_BILL_TRANS_OR><TOT_BILL_AMT_OR></TOT_BILL_AMT_OR><CANCEL_ID></CANCEL_ID><CANCEL_ID></CANCEL_ID><CUST_NAME>{ref.get('CUST_NAME', '')}</CUST_NAME><CUST_ADDR_1>{ref.get('CUST_ADDR_1', '')}</CUST_ADDR_1><CUST_ADDR_2>{ref.get('CUST_ADDR_2', '')}</CUST_ADDR_2><CUST_ADDR_3>{ref.get('CUST_ADDR_3', '')}</CUST_ADDR_3><CUST_PHONE_NO>{ref.get('CUST_PHONE_NO', '')}</CUST_PHONE_NO></DATA></SERVICE_BOX></HQ_REQUEST>"""

    @staticmethod
    def build_print(tx: TransactionContext) -> str:
        ref = tx.ref_data
        return f"""<?xml version="1.0" encoding="UTF-8"?><HQ_REQUEST><SERVICE_BOX><ADDRESS><VENDOR_ID>{tx.store.vendor_id}</VENDOR_ID><SERVICE_ID>{tx.store.service_id}</SERVICE_ID><METHOD>REPRINTSLIP</METHOD></ADDRESS><DATA><PAYMENT_CHANNEL>{tx.store.payment_channel}</PAYMENT_CHANNEL><VENDOR_ID>{ref.get('VENDOR_ID', '')}</VENDOR_ID><SERV_ID>{ref.get('SERV_ID', '')}</SERV_ID><SERVICE_ID>{ref.get('SERV_ID', '')}</SERVICE_ID><STORE_ID>{tx.store.store_id}</STORE_ID><STATION_ID>1</STATION_ID><BUS_DATE>{tx.bus_date}</BUS_DATE><BUS_TIME>{tx.bus_time}</BUS_TIME><COMMON_TRN_ID>{tx.common_trn_id}</COMMON_TRN_ID><SEQ_NO>{ref.get('SEQ_NO', '')}</SEQ_NO><CLIENT_SERV_SEQ>{ref.get('SEQ_NO', '')}</CLIENT_SERV_SEQ><SHIFT_ID>9</SHIFT_ID><TRANS_TYPE>N</TRANS_TYPE><ACCT_NO></ACCT_NO><BILL_AMT>{ref.get('BILL_AMT', '')}</BILL_AMT><ROUND_BILL_AMT>{ref.get('BILL_AMT', '')}</ROUND_BILL_AMT><VAT_AMT>{tx.store.vat_amt}</VAT_AMT><REPT_TYPE>{tx.store.rept_type}</REPT_TYPE><TX_ID>{ref.get('TX_ID', '')}</TX_ID><REPT_NO></REPT_NO><PREV_REF_SEQ></PREV_REF_SEQ><PREV_REF_DATE></PREV_REF_DATE><SERV_CHARGE_NO></SERV_CHARGE_NO><ITEM_NAME>{tx.store.item_name}</ITEM_NAME><ITEM_SELECTION>N</ITEM_SELECTION><EMPLOYEE_ID>{tx.store.employee_id}</EMPLOYEE_ID><POS_TAX_ID>{tx.store.pos_tax_id}</POS_TAX_ID><DATA_1>{ref.get('DATA_1', '')}</DATA_1><DATA_2>{ref.get('DATA_2', '')}</DATA_2><DATA_3>{ref.get('DATA_3', '')}</DATA_3><DATA_4>{ref.get('DATA_4', '')}</DATA_4><DATA_5>{ref.get('DATA_5', '')}</DATA_5><DATA_6>{ref.get('DATA_6', '')}</DATA_6><DATA_7>{ref.get('DATA_7', '')}</DATA_7><DATA_9>{ref.get('DATA_9', '')}</DATA_9><ZONE>{tx.store.zone}</ZONE><CANCEL_ID></CANCEL_ID></DATA></SERVICE_BOX></HQ_REQUEST>"""

    @staticmethod
    def build_or(tx: TransactionContext) -> str:
        ref = tx.ref_data
        return f"""<?xml version="1.0" encoding="UTF-8"?><HQ_REQUEST><SERVICE_BOX><ADDRESS><VENDOR_ID>{tx.store.vendor_id}</VENDOR_ID><SERVICE_ID>{tx.store.service_id}</SERVICE_ID><METHOD>OR</METHOD></ADDRESS><DATA><PAYMENT_CHANNEL>{tx.store.payment_channel}</PAYMENT_CHANNEL><VENDOR_ID>{ref.get('VENDOR_ID', '')}</VENDOR_ID><SERVICE_ID>{ref.get('SERV_ID', '')}</SERVICE_ID><SERV_ID>{ref.get('SERV_ID', '')}</SERV_ID><STORE_ID>{tx.store.store_id}</STORE_ID><STATION_ID>1</STATION_ID><BUS_DATE>{tx.bus_date}</BUS_DATE><BUS_TIME>{tx.bus_time}</BUS_TIME><SYS_DATE>{tx.bus_date}</SYS_DATE><SYS_TIME>{tx.bus_time}</SYS_TIME><TX_ID>{ref.get('TX_ID', '')}</TX_ID><BILL_AMT>{ref.get('BILL_AMT', '')}</BILL_AMT><ROUND_BILL_AMT>{ref.get('BILL_AMT', '')}</ROUND_BILL_AMT><VAT_AMT>{tx.store.vat_amt}</VAT_AMT><PAYMENT_TYPE>001</PAYMENT_TYPE><CANCEL_ID></CANCEL_ID></DATA></SERVICE_BOX></HQ_REQUEST>"""

    @staticmethod
    def build_or_cancel(tx: TransactionContext) -> str:
        ref = tx.ref_data
        return f"""<?xml version="1.0" encoding="UTF-8"?><HQ_REQUEST><SERVICE_BOX><ADDRESS><VENDOR_ID>{tx.store.vendor_id}</VENDOR_ID><SERVICE_ID>{tx.store.service_id}</SERVICE_ID><METHOD>ORCancel</METHOD></ADDRESS><DATA><PAYMENT_CHANNEL>{tx.store.payment_channel}</PAYMENT_CHANNEL><VENDOR_ID>{ref.get('VENDOR_ID', '')}</VENDOR_ID><SERVICE_ID>{ref.get('SERV_ID', '')}</SERVICE_ID><SERV_ID>{ref.get('SERV_ID', '')}</SERV_ID><STORE_ID>{tx.store.store_id}</STORE_ID><STATION_ID>1</STATION_ID><BUS_DATE>{tx.bus_date}</BUS_DATE><BUS_TIME>{tx.bus_time}</BUS_TIME><TX_ID>{ref.get('TX_ID', '')}</TX_ID><PAYMENT_TYPE>001</PAYMENT_TYPE><CANCEL_ID></CANCEL_ID></DATA></SERVICE_BOX></HQ_REQUEST>"""

    @staticmethod
    def build_or_confirm(tx: TransactionContext) -> str:
        ref = tx.ref_data
        return f"""<?xml version="1.0" encoding="UTF-8"?><HQ_REQUEST><SERVICE_BOX><ADDRESS><VENDOR_ID>{tx.store.vendor_id}</VENDOR_ID><SERVICE_ID>{tx.store.service_id}</SERVICE_ID><METHOD>ORConfirm</METHOD></ADDRESS><DATA><PAYMENT_CHANNEL>{tx.store.payment_channel}</PAYMENT_CHANNEL><VENDOR_ID>{ref.get('VENDOR_ID', '')}</VENDOR_ID><SERVICE_ID>{ref.get('SERV_ID', '')}</SERVICE_ID><SERV_ID>{ref.get('SERV_ID', '')}</SERV_ID><STATION_ID>1</STATION_ID><STORE_ID>{tx.store.store_id}</STORE_ID><BUS_DATE>{tx.bus_date}</BUS_DATE><BUS_TIME>{tx.bus_time}</BUS_TIME><BILL_AMT>{ref.get('BILL_AMT', '')}</BILL_AMT><ROUND_BILL_AMT>{ref.get('BILL_AMT', '')}</ROUND_BILL_AMT><VAT_AMT>{tx.store.vat_amt}</VAT_AMT><TX_ID>{ref.get('TX_ID', '')}</TX_ID><SEQ_NO>{ref.get('SUM_SEQ', '')}</SEQ_NO><CLIENT_SERV_SEQ>{ref.get('SUM_SEQ', '')}</CLIENT_SERV_SEQ><SERV_ID>{ref.get('SERV_ID', '')}</SERV_ID><DATA_1>{ref.get('DATA_1', '')}</DATA_1><DATA_2>{ref.get('DATA_2', '')}</DATA_2><DATA_3>{ref.get('DATA_3', '')}</DATA_3><DATA_4>{ref.get('DATA_4', '')}</DATA_4><DATA_5>{ref.get('DATA_5', '')}</DATA_5><DATA_6>{ref.get('DATA_6', '')}</DATA_6><DATA_7>{ref.get('DATA_7', '')}</DATA_7><DATA_9>{ref.get('DATA_9', '')}</DATA_9><ZONE>{tx.store.zone}</ZONE><PAYMENT_TYPE>001</PAYMENT_TYPE><TOT_BILL_TRANS></TOT_BILL_TRANS><TOT_BILL_AMT></TOT_BILL_AMT><TOT_VENDOR_TRANS></TOT_VENDOR_TRANS><TOT_VENDOR_AMT></TOT_VENDOR_AMT><TOT_COUNTER_TRANS></TOT_COUNTER_TRANS><TOT_COUNTER_AMT></TOT_COUNTER_AMT><TOT_CLIENT_TRANS></TOT_CLIENT_TRANS><TOT_CLIENT_AMT></TOT_CLIENT_AMT><TOT_BILL_TRANS_OR></TOT_BILL_TRANS_OR><TOT_BILL_AMT_OR></TOT_BILL_AMT_OR><CANCEL_ID></CANCEL_ID></DATA></SERVICE_BOX></HQ_REQUEST>"""

    @staticmethod
    def build_amt_confirm(tx: TransactionContext) -> str:
        ref = tx.ref_data
        return f"""<?xml version="1.0" encoding="UTF-8"?><HQ_REQUEST><SERVICE_BOX><ADDRESS><VENDOR_ID>{tx.store.vendor_id}</VENDOR_ID><SERVICE_ID>{tx.store.service_id}</SERVICE_ID><METHOD>DataExchangeConfirm</METHOD></ADDRESS><DATA><PAYMENT_CHANNEL>{tx.store.payment_channel}</PAYMENT_CHANNEL><VENDOR_ID>{ref.get('VENDOR_ID', '')}</VENDOR_ID><SERV_ID>{ref.get('SERV_ID', '')}</SERV_ID><SERVICE_ID>{ref.get('SERV_ID', '')}</SERVICE_ID><STATION_ID>1</STATION_ID><STORE_ID>{tx.store.store_id}</STORE_ID><BUS_DATE>{tx.bus_date}</BUS_DATE><BUS_TIME>{tx.bus_time}</BUS_TIME><SYS_DATE>{tx.bus_date}</SYS_DATE><SYS_TIME>{tx.bus_time}</SYS_TIME><TX_ID>{ref.get('TX_ID', '')}</TX_ID><SEQ_NO>{ref.get('SEQ_NO', '')}</SEQ_NO><EMPLOYEE_ID>{tx.store.employee_id}</EMPLOYEE_ID><CLIENT_SERV_SEQ>{ref.get('SEQ_NO', '')}</CLIENT_SERV_SEQ><SERV_ID>{tx.store.service_id}</SERV_ID><BILL_AMT>{ref.get('SUM_AMT', '')}</BILL_AMT><ROUND_BILL_AMT>{ref.get('SUM_AMT', '')}</ROUND_BILL_AMT><ACCT_NO></ACCT_NO><VAT_AMT>{tx.store.vat_amt}</VAT_AMT><DATA_1>{ref.get('DATA_1', '')}</DATA_1><DATA_2>{ref.get('DATA_2', '')}</DATA_2><DATA_3>{ref.get('DATA_3', '')}</DATA_3><DATA_4>{ref.get('DATA_4', '')}</DATA_4><DATA_5>{ref.get('DATA_5', '')}</DATA_5><DATA_6>{ref.get('DATA_6', '')}</DATA_6><DATA_7>{ref.get('DATA_7', '')}</DATA_7><DATA_9>{ref.get('DATA_9', '')}</DATA_9><ZONE>{tx.store.zone}</ZONE><PAYMENT_TYPE>001</PAYMENT_TYPE><TOT_BILL_TRANS></TOT_BILL_TRANS><TOT_BILL_AMT></TOT_BILL_AMT><TOT_VENDOR_TRANS></TOT_VENDOR_TRANS><TOT_VENDOR_AMT></TOT_VENDOR_AMT><TOT_COUNTER_TRANS></TOT_COUNTER_TRANS><TOT_COUNTER_AMT></TOT_COUNTER_AMT><TOT_CLIENT_TRANS></TOT_CLIENT_TRANS><TOT_CLIENT_AMT></TOT_CLIENT_AMT><TOT_BILL_TRANS_OR></TOT_BILL_TRANS_OR><TOT_BILL_AMT_OR></TOT_BILL_AMT_OR><CANCEL_ID></CANCEL_ID><CANCEL_ID></CANCEL_ID><CUST_NAME>{ref.get('CUST_NAME', '')}</CUST_NAME><CUST_ADDR_1>{ref.get('CUST_ADDR_1', '')}</CUST_ADDR_1><CUST_ADDR_2>{ref.get('CUST_ADDR_2', '')}</CUST_ADDR_2><CUST_ADDR_3>{ref.get('CUST_ADDR_3', '')}</CUST_ADDR_3><CUST_PHONE_NO>{ref.get('CUST_PHONE_NO', '')}</CUST_PHONE_NO></DATA></SERVICE_BOX></HQ_REQUEST>"""

    @staticmethod
    def build_std_tk_inquiry(tx: TransactionContext) -> str:
        return f"""<?xml version="1.0" encoding="UTF-8"?><HQ_REQUEST><SERVICE_BOX><ADDRESS><VENDOR_ID>{tx.store.vendor_id}</VENDOR_ID><SERVICE_ID>{tx.store.service_id}</SERVICE_ID><METHOD>StdTkInquiry</METHOD></ADDRESS><DATA><PAYMENT_CHANNEL>{tx.store.payment_channel}</PAYMENT_CHANNEL><VENDOR_ID>{tx.store.vendor_id}</VENDOR_ID><SERV_ID>{tx.store.service_id}</SERV_ID><SERVICE_ID>{tx.store.service_id}</SERVICE_ID><STORE_ID>{tx.store.store_id}</STORE_ID><STATION_ID>1</STATION_ID><BUS_DATE>{tx.bus_date}</BUS_DATE><BUS_TIME>{tx.bus_time}</BUS_TIME><SYS_DATE>{tx.bus_date}</SYS_DATE><SYS_TIME>{tx.bus_time}</SYS_TIME><COMMON_TRN_ID>{tx.common_trn_id}</COMMON_TRN_ID><SEQ_NO></SEQ_NO><CLIENT_SERV_SEQ></CLIENT_SERV_SEQ><SHIFT_ID></SHIFT_ID><TRANS_TYPE></TRANS_TYPE><ACCT_NO></ACCT_NO><BILL_AMT>{tx.bill.bill_amt}</BILL_AMT><ROUND_BILL_AMT>{tx.bill.bill_amt}</ROUND_BILL_AMT><VAT_AMT>{tx.store.vat_amt}</VAT_AMT><REPT_TYPE>{tx.store.rept_type}</REPT_TYPE><REPT_NO></REPT_NO><PREV_REF_SEQ></PREV_REF_SEQ><PREV_REF_DATE></PREV_REF_DATE><SERV_CHARGE_NO></SERV_CHARGE_NO><ITEM_NAME>{tx.store.item_name}</ITEM_NAME><ITEM_SELECTION></ITEM_SELECTION><EMPLOYEE_ID>{tx.store.employee_id}</EMPLOYEE_ID><POS_TAX_ID>{tx.store.pos_tax_id}</POS_TAX_ID><DATA_1>{tx.data_1}</DATA_1><DATA_2>{tx.data_2}</DATA_2><DATA_3>{tx.data_3}</DATA_3><DATA_4>{tx.data_4}</DATA_4><DATA_5>{tx.data_5}</DATA_5><DATA_6>{tx.data_6}</DATA_6><DATA_7>{tx.data_7}</DATA_7><DATA_9>{tx.data_9}</DATA_9><ZONE>{tx.store.zone}</ZONE><PAYMENT_TYPE>001</PAYMENT_TYPE><CANCEL_ID></CANCEL_ID><CUST_NAME>{tx.customer.name}</CUST_NAME><CUST_ADDR_1>{tx.customer.addr_1}</CUST_ADDR_1><CUST_ADDR_2>{tx.customer.addr_2}</CUST_ADDR_2><CUST_ADDR_3>{tx.customer.addr_3}</CUST_ADDR_3><CUST_PHONE_NO>{tx.customer.phone_no}</CUST_PHONE_NO></DATA></SERVICE_BOX></HQ_REQUEST>"""

    @staticmethod
    def build_inquiry(tx: TransactionContext) -> str:
        return f"""<?xml version="1.0" encoding="UTF-8"?><HQ_REQUEST><SERVICE_BOX><ADDRESS><VENDOR_ID>{tx.store.vendor_id}</VENDOR_ID><SERVICE_ID>{tx.store.service_id}</SERVICE_ID><METHOD>Inquiry</METHOD></ADDRESS><DATA><PAYMENT_CHANNEL>{tx.store.payment_channel}</PAYMENT_CHANNEL><VENDOR_ID>{tx.store.vendor_id}</VENDOR_ID><SERV_ID>{tx.store.service_id}</SERV_ID><SERVICE_ID>{tx.store.service_id}</SERVICE_ID><STORE_ID>{tx.store.store_id}</STORE_ID><STATION_ID>1</STATION_ID><BUS_DATE>{tx.bus_date}</BUS_DATE><BUS_TIME>{tx.bus_time}</BUS_TIME><SYS_DATE>{tx.bus_date}</SYS_DATE><SYS_TIME>{tx.bus_time}</SYS_TIME><COMMON_TRN_ID>{tx.common_trn_id}</COMMON_TRN_ID><SEQ_NO/><CLIENT_SERV_SEQ/><SHIFT_ID></SHIFT_ID><TRANS_TYPE>N</TRANS_TYPE><ACCT_NO/><BILL_AMT>{tx.bill.bill_amt}</BILL_AMT><ROUND_BILL_AMT/><VAT_AMT>{tx.store.vat_amt}</VAT_AMT><REPT_TYPE>{tx.store.rept_type}</REPT_TYPE><REPT_NO/><PREV_REF_SEQ/><PREV_REF_DATE/><SERV_CHARGE_NO/><ITEM_NAME>{tx.store.item_name}</ITEM_NAME><ITEM_SELECTION>N</ITEM_SELECTION><EMPLOYEE_ID>{tx.store.employee_id}</EMPLOYEE_ID><POS_TAX_ID/><DATA_1>{tx.data_1}</DATA_1><DATA_2>{tx.data_2}</DATA_2><DATA_3>{tx.data_3}</DATA_3><DATA_4>{tx.data_4}</DATA_4><DATA_5>{tx.data_5}</DATA_5><DATA_6>{tx.data_6}</DATA_6><DATA_7>{tx.data_7}</DATA_7><DATA_9>{tx.data_9}</DATA_9><ZONE>{tx.store.zone}</ZONE><PAYMENT_TYPE>001</PAYMENT_TYPE><CANCEL_ID/></DATA></SERVICE_BOX></HQ_REQUEST>"""

# ==========================================
# 3. Network & API Client (ตัวจัดการ API)
# ==========================================

class SoapApiClient:
    def __init__(self, endpoint_url: str):
        self.endpoint_url = endpoint_url

    def _wrap_soap_envelope(self, action_payload: str) -> str:
        return f"""<soapenv:Envelope xmlns:soapenv="http://schemas.xmlsoap.org/soap/envelope/" xmlns:por="http://portal.cs/">
        <soapenv:Header/>
        <soapenv:Body><por:CSService><arg0><![CDATA[{action_payload}]]></arg0></por:CSService></soapenv:Body>
        </soapenv:Envelope>"""

    def send_request(self, xml_payload: str) -> str:
        """ส่งข้อมูลและถอดรหัส Base64 Return ให้พร้อมใช้งาน"""
        soap_data = self._wrap_soap_envelope(xml_payload)
        headers = {'Content-Type': 'text/xml'}
        
        try:
            response = requests.post(self.endpoint_url, headers=headers, data=soap_data.encode("utf-8"))
            response.raise_for_status()
            
            # Extract CDATA and Decode Base64
            encoded_val = response.text.split("<return>")[-1].split("</return>")[0]
            decoded_xml = base64.b64decode(encoded_val).decode('utf-8')
            return decoded_xml
            
        except requests.exceptions.RequestException as e:
            print(f"❌ API Connection Error: {e}")
            return ""

# ==========================================
# 4. Response Parser (จัดการข้อมูลขากลับ)
# ==========================================

class ResponseParser:
    FIELDS = ['SUCCESS', 'CODE', 'DESCRIPTOR', 'VENDOR_ID', 'SERV_ID', 'TX_ID', 'PRINTSLIP', 'VAT', 'BILL_AMT', 'FEE', 'FEE_VAT', 'DATA_1', 'DATA_2', 'DATA_3', 'DATA_4', 'DATA_5', 'DATA_6', 'DATA_7', 'DATA_9', 'CUSTOMER_NAME', 'CUSTOMER_ADDR_1', 'CUSTOMER_ADDR_2', 'CUSTOMER_ADDR_3', 'CUSTOMER_TEL_NO', 'ACCT_NO']

    @staticmethod
    def parse_to_dict(xml_response: str) -> Dict[str, Any]:
        """แปลง XML เป็น Dictionary ให้ใช้งานต่อได้ง่าย"""
        if not xml_response: return {}
        try:
            parsed_data = xd.parse(xml_response)
            hq_response = parsed_data.get('HQ_RESPONSE', {})
            return {field: hq_response.get(field, '') for field in ResponseParser.FIELDS}
        except Exception as e:
            print(f"❌ Parse Error: {e}")
            return {}

    @staticmethod
    def display_as_dataframe(response_dict: Dict[str, Any]):
        """แสดงผลเฉพาะ Field ที่มีข้อมูลผ่าน Pandas (ทดแทน Dataframe ตัวเดิม)"""
        filtered_dict = {k: v for k, v in response_dict.items() if v is not None and str(v).strip() != ''}
        df = pd.DataFrame([filtered_dict])
        display(df)  # ใช้ได้ทันทีหากรันใน Jupyter/Colab

# ==========================================
# 5. Database Connector (จัดการเชื่อมต่อ Oracle)
# ==========================================

class DatabaseConnector:
    def __init__(self, ip='10.182.236.52', service_name='ONLPRD'):
        self.dsn = oracledb.makedsn(ip, '1521', service_name=service_name)
        self.user = 'CS_DEV'
        self.pwd = '1234'

    def execute_query(self, sql_query: str):
        """ใช้ Context Manager (with) เพื่อให้มั่นใจว่า Connection จะถูกปิดเสมอเมื่อใช้งานเสร็จ"""
        try:
            with oracledb.connect(user=self.user, password=self.pwd, dsn=self.dsn) as conn:
                with conn.cursor() as cursor:
                    cursor.execute(sql_query)
                    
                    if sql_query.strip().upper().startswith("SELECT"):
                        result = cursor.fetchall()
                        return result[-1] if len(result) == 1 else result
                    else:
                        conn.commit()
                        print("✅ อัพเดทข้อมูลใน Database สำเร็จ")
                        return True
        except Exception as e:
            print(f"❌ Database Error: {e}")
            return None


class OracleManager(DatabaseConnector):
    """คลาสสำหรับจัดการระบบตรวจสอบฐานข้อมูล (Pre-Check & Post-Check)"""
    
    def get_dataframe(self, query: str, params: dict = None) -> pd.DataFrame:
        """Helper Function: ดึงข้อมูลจาก Database และแปลงเป็น Pandas DataFrame ทันที"""
        try:
            with oracledb.connect(user=self.user, password=self.pwd, dsn=self.dsn) as conn:
                # ใช้ pandas read_sql ช่วยให้ทำงานกับข้อมูลแบบตาราง (Excel/JSON) ได้ง่ายขึ้นมาก
                df = pd.read_sql(query, con=conn, params=params)
                return df
        except Exception as e:
            print(f"❌ Database Query Error: {e}")
            return pd.DataFrame()

    # ==========================================
    # Phase 1: Pre-Check (ตรวจสอบ Config 5 ตาราง)
    # ==========================================
    def check_vendor_config(self, vendor_id: str, service_id: str, action: Literal['show', 'export'] = 'show') -> Any:
        """ตรวจสอบ Config 5 ตาราง และเลือก Output เป็น JSON ('show') หรือไฟล์ Excel ('export')"""
        
        # ใช้ Bind Variable (:vendor_id) แทน f-string มาตรฐานระดับ Enterprise
        params = {"vendor_id": vendor_id, "service_id": service_id}
        
        # กำหนด 5 ตารางที่ต้องการตรวจสอบ (ปรับแก้ชื่อตารางที่ 4 และ 5 ได้ตามโครงสร้างจริงของคุณ)
        queries = {
            "Client_Config": "SELECT * FROM ONLSTD.WS_CLIENT_CONFIG WHERE VENDOR_ID = :vendor_id AND SERVICE_ID = :service_id",
            "Charge_Step": "SELECT * FROM ONLSTD.WS_CLIENT_CHARGE_STEP WHERE VENDOR_ID = :vendor_id AND SERVICE_ID = :service_id",
            "Reprint_Limit": "SELECT * FROM ONLSTD.WS_CLIENT_REPRINT WHERE VENDOR_ID = :vendor_id AND SERVICE_ID = :service_id",
            "AutoFix_Tx": "SELECT * FROM ONLSTD.WS_CLIENT_AUTOFIXTX WHERE VENDOR_ID = :vendor_id AND SERVICE_ID = :service_id",
            "Vendor_Master": "SELECT * FROM ONLSTD.WS_VENDOR_MASTER WHERE VENDOR_ID = :vendor_id" # อ้างอิงด้วย vendor_id อย่างเดียว
        }
        
        # Query ข้อมูลทั้งหมดเก็บไว้ในรูปแบบ Dictionary ของ DataFrame
        print(f"🔍 [Pre-Check] กำลังตรวจสอบ Config ของ Vendor: {vendor_id}...")
        results_df = {name: self.get_dataframe(sql, params) for name, sql in queries.items()}
        
        if action == 'show':
            # แปลง DataFrame เป็น JSON Dictionary
            json_result = {name: json.loads(df.to_json(orient='records')) for name, df in results_df.items()}
            print("✅ [Show Mode] สรุปข้อมูลรูปแบบ JSON สำเร็จ")
            return json_result
            
        elif action == 'export':
            # บันทึกแต่ละตารางลงคนละ Sheet ในไฟล์ Excel (.xlsx) เดียวกัน
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            file_name = f"VendorConfig_{vendor_id}_{timestamp}.xlsx"
            
            try:
                # จำเป็นต้องใช้ไลบรารี openpyxl หรือ xlsxwriter (pip install openpyxl)
                with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
                    for name, df in results_df.items():
                        df.to_excel(writer, sheet_name=name, index=False)
                print(f"✅ [Export Mode] สร้างและบันทึกไฟล์ Excel สำเร็จ: {file_name}")
                return file_name
            except Exception as e:
                print(f"❌ Error Exporting Excel: {e}")
                return None

    # ==========================================
    # Phase 2: Post-Check (ค้นหา TX_ID หลังทำรายการ)
    # ==========================================
    def get_transaction_by_tx_id(self, tx_id: str) -> pd.DataFrame:
        """นำ TX_ID ที่ได้จาก Response มาค้นหาใน Database เพื่อยืนยันข้อมูล"""
        if not tx_id:
            print("⚠️ ไม่พบ TX_ID สำหรับใช้ค้นหาข้อมูล")
            return pd.DataFrame()
            
        print(f"🔍 [Post-Check] กำลังค้นหา Transaction: {tx_id} ในระบบ...")
        
        # มีการปรับ SQL เพื่อแสดงค่า STORE_ID และแสดง PAY เมื่อ TS TD มีข้อมูล
        query = """
            SELECT 
                TX_ID, 
                STORE_ID,
                CASE WHEN TS IS NOT NULL AND TD IS NOT NULL THEN PAY ELSE NULL END AS PAY_INFO,
                SYSTEM_DATE_TIME,
                Tbl.*
            FROM ONLSTD.WS_ONLINE_TX Tbl
            WHERE TX_ID = :tx_id OR R_SERVICE_RUNNO = :tx_id
        """
        params = {"tx_id": tx_id}
        
        df = self.get_dataframe(query, params)
        if df.empty:
            print("⚠️ ค้นหาสำเร็จ แต่ยังไม่พบข้อมูล Transaction นี้บันทึกลงใน Database")
        else:
            print("✅ พบข้อมูล Transaction ใน Database แล้ว")
        return df

In [39]:
URL = "http://qacspos.counterservice.co.th:80/DCWSCDSONLINE/WSCDSService"

# 1. สร้างและตั้งค่า Transaction 
tx = TransactionContext(data_1=10, data_2=13)
tx.store.store_id = "09884"
tx.store.vendor_id = "0994000160127"
tx.store.service_id = "00"
tx.bill.bill_amt = "80"

tx.prepare_data() # แปลงเลข Gen/List ให้พร้อมใช้งาน

# 2. จัดเตรียมเครื่องมือติดต่อ API
api_client = SoapApiClient(endpoint_url=URL)

# --- ACTION 1: DataExchange ---
print("▶️ กำลังส่ง DataExchange...")
exchange_xml = SoapPayloadBuilder.build_data_exchange(tx)
response_xml = api_client.send_request(exchange_xml)

res_dict = ResponseParser.parse_to_dict(response_xml)
ResponseParser.display_as_dataframe(res_dict)

print(res_dict)
if res_dict.get('CODE') == "100":  
    print("\n▶️ กำลังส่ง Cancel ต่อเนื่อง...")
    tx.load_reference_from_response(res_dict) # โหลดค่าอ้างอิงจากรอบแรกมาเก็บไว้
    
    cancel_xml = SoapPayloadBuilder.build_cancel(tx)
    cancel_res_xml = api_client.send_request(cancel_xml)
    
    cancel_res_dict = ResponseParser.parse_to_dict(cancel_res_xml)
    ResponseParser.display_as_dataframe(cancel_res_dict)

▶️ กำลังส่ง DataExchange...


,SUCCESS,CODE,DESCRIPTOR,VENDOR_ID,SERV_ID,TX_ID,BILL_AMT,FEE,FEE_VAT,DATA_1,DATA_2
0,true,100,success,0994000160127,00,14678513,80,0,0,2402252121,0270499407028


{'SUCCESS': 'true', 'CODE': '100', 'DESCRIPTOR': 'success', 'VENDOR_ID': '0994000160127', 'SERV_ID': '00', 'TX_ID': '14678513', 'PRINTSLIP': None, 'VAT': None, 'BILL_AMT': '80', 'FEE': '0', 'FEE_VAT': '0', 'DATA_1': '2402252121', 'DATA_2': '0270499407028', 'DATA_3': None, 'DATA_4': None, 'DATA_5': None, 'DATA_6': None, 'DATA_7': None, 'DATA_9': '', 'CUSTOMER_NAME': None, 'CUSTOMER_ADDR_1': None, 'CUSTOMER_ADDR_2': None, 'CUSTOMER_ADDR_3': None, 'CUSTOMER_TEL_NO': None, 'ACCT_NO': None}

▶️ กำลังส่ง Cancel ต่อเนื่อง...


,SUCCESS,CODE,DESCRIPTOR
0,true,100,success


In [37]:

URL = "http://qacspos.counterservice.co.th:80/DCWSCDSONLINE/WSCDSService"
db_manager = OracleManager(ip='10.182.236.52', service_name='ONLPRD')

# ------------------------------------------
# STEP 1: Pre-Check (เช็ค Config ก่อนเริ่มงาน)
# ------------------------------------------
vendor_to_check = "0994000160127"
service_to_check = "00"

# ให้ User เลือกว่าจะ 'show' หรือ 'export'
action_type = 'export' # เปลี่ยนเป็น 'show' ถ้าต้องการดู JSON

config_data = db_manager.check_vendor_config(
    vendor_id=vendor_to_check, 
    service_id=service_to_check, 
    action=action_type
)

# ถ้าเลือก 'show' ลองปริ้นข้อมูลออกมาดู
if action_type == 'show':
    print(json.dumps(config_data, indent=4, ensure_ascii=False))

# ------------------------------------------
# STEP 2: เตรียมและยิง API
# ------------------------------------------
tx = TransactionContext(data_1=10, data_2=13)
tx.store.vendor_id = vendor_to_check
tx.store.service_id = service_to_check
tx.prepare_data()

api_client = SoapApiClient(endpoint_url=URL)

print("\n▶️ กำลังส่ง DataExchange API...")
exchange_xml = SoapPayloadBuilder.build_data_exchange(tx)
response_xml = api_client.send_request(exchange_xml)

# แปลง Response XML เป็น Dict
res_dict = ResponseParser.parse_to_dict(response_xml)
ResponseParser.display_as_dataframe(res_dict)

# ------------------------------------------
# STEP 3: Post-Check (ตรวจเช็ค Database หลังทำรายการ)
# ------------------------------------------
if res_dict.get('CODE') == "100":
    returned_tx_id = res_dict.get('TX_ID')
    
    # ถ้ามี TX_ID กลับมา ให้นำไป Query ตาราง WS_ONLINE_TX ต่อทันที
    if returned_tx_id:
        tx_data_df = db_manager.get_transaction_by_tx_id(tx_id=returned_tx_id[:8])
        display(tx_data_df) # แสดงผลข้อมูลที่บันทึกลง Oracle สำเร็จ

🔍 [Pre-Check] กำลังตรวจสอบ Config ของ Vendor: 0994000160127...


C:\Users\nakarinsue\AppData\Local\Temp\ipykernel_3944\4122365882.py:311: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=conn, params=params)
C:\Users\nakarinsue\AppData\Local\Temp\ipykernel_3944\4122365882.py:311: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=conn, params=params)
C:\Users\nakarinsue\AppData\Local\Temp\ipykernel_3944\4122365882.py:311: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=conn, params=params)


❌ Database Query Error: DPY-4008: no bind placeholder named ":service_id" was found in the SQL text
✅ [Export Mode] สร้างและบันทึกไฟล์ Excel สำเร็จ: VendorConfig_0994000160127_20260311_214349.xlsx

▶️ กำลังส่ง DataExchange API...


,SUCCESS,CODE,DESCRIPTOR,VENDOR_ID,SERV_ID,TX_ID,BILL_AMT,FEE,FEE_VAT,DATA_1,DATA_2
0,true,100,success,0994000160127,00,14678512,50,0,0,0232506956,5916867125880


🔍 [Post-Check] กำลังค้นหา Transaction: 14678512 ในระบบ...
❌ Database Query Error: ORA-00904: "PAY": invalid identifier
Help: https://docs.oracle.com/error-help/db/ora-00904/
⚠️ ค้นหาสำเร็จ แต่ยังไม่พบข้อมูล Transaction นี้บันทึกลงใน Database


C:\Users\nakarinsue\AppData\Local\Temp\ipykernel_3944\4122365882.py:311: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=conn, params=params)


""


In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel
from typing import Optional, Dict, Any
import os
import json

# นำเข้าคลาสต่างๆ ที่เราสร้างไว้จากไฟล์หลัก (สมมติว่าชื่อ core_logic.py)
# from core_logic import TransactionContext, SoapPayloadBuilder, SoapApiClient, ResponseParser, OracleManager

app = FastAPI(title="Enterprise POS Integration API", version="1.0.0")

# ==========================================
# 1. Pydantic Models (กำหนดรูปแบบ JSON Input)
# ==========================================

class ActionRequest(BaseModel):
    """JSON สำหรับยิง Action API"""
    url: str = "http://localhost:80/Test/WSCDSService"
    vendor_id: str
    service_id: str
    store_id: str = "09892"
    bill_amt: str = "0"
    data_1: Optional[str] = ""
    data_3: Optional[str] = ""
    # ใช้สำหรับกรณี Cancel, Confirm ที่ต้องอ้างอิงข้อมูลเดิม
    ref_data: Optional[Dict[str, Any]] = {} 

class ConfigRequest(BaseModel):
    """JSON สำหรับเช็ค Config (Show/Export)"""
    vendor_id: str
    service_id: str

class TxIdRequest(BaseModel):
    """JSON สำหรับค้นหา Transaction"""
    tx_id: str

# สร้าง Instance พื้นฐาน
db_manager = OracleManager(ip='10.182.236.52', service_name='ONLPRD')

# ==========================================
# เส้นที่ 1: API สำหรับ Action ต่างๆ (1 เส้นต่อ 1 Action)
# ==========================================

def prepare_tx_from_request(req: ActionRequest) -> TransactionContext:
    """Helper สำหรับแปลง JSON Request เป็น TransactionContext"""
    tx = TransactionContext(data_1=req.data_1, data_3=req.data_3)
    tx.store.vendor_id = req.vendor_id
    tx.store.service_id = req.service_id
    tx.store.store_id = req.store_id
    tx.bill.bill_amt = req.bill_amt
    
    if req.ref_data:
        # ถ้ามีการส่ง ref_data มาด้วย (เช่น Cancel) ให้นำไปโหลดใส่ tx
        tx.load_reference_from_response(req.ref_data)
        
    tx.prepare_data()
    return tx

def process_soap_action(action_type: str, req: ActionRequest) -> Dict[str, Any]:
    tx = prepare_tx_from_request(req)
    api_client = SoapApiClient(endpoint_url=req.url)
    
    # เลือก Builder ตาม Action
    builders = {
        "data_exchange": SoapPayloadBuilder.build_data_exchange,
        "cancel": SoapPayloadBuilder.build_cancel,
        "exchange_confirm": SoapPayloadBuilder.build_data_exchange_confirm,
        "print": SoapPayloadBuilder.build_print,
        "or": SoapPayloadBuilder.build_or,
        "or_cancel": SoapPayloadBuilder.build_or_cancel,
        "or_confirm": SoapPayloadBuilder.build_or_confirm,
        "amt_confirm": SoapPayloadBuilder.build_amt_confirm,
        "std_tk_inquiry": SoapPayloadBuilder.build_std_tk_inquiry,
        "inquiry": SoapPayloadBuilder.build_inquiry,
    }
    
    if action_type not in builders:
        raise HTTPException(status_code=400, detail="Action ไม่ถูกต้อง")
        
    xml_payload = builders[action_type](tx)
    response_xml = api_client.send_request(xml_payload)
    
    if not response_xml:
        raise HTTPException(status_code=500, detail="ไม่สามารถเชื่อมต่อ SOAP Service ได้ หรือไม่มี Response")
        
    return ResponseParser.parse_to_dict(response_xml)

# สร้าง Endpoint แยกร่าย Action
@app.post("/api/action/data_exchange", summary="ยิง Action: DataExchange")
def action_data_exchange(req: ActionRequest):
    return process_soap_action("data_exchange", req)

@app.post("/api/action/cancel", summary="ยิง Action: Cancel")
def action_cancel(req: ActionRequest):
    return process_soap_action("cancel", req)

@app.post("/api/action/exchange_confirm", summary="ยิง Action: DataExchangeConfirm")
def action_exchange_confirm(req: ActionRequest):
    return process_soap_action("exchange_confirm", req)

# (สามารถเพิ่ม @app.post สำหรับ Print, OR, Inquiry ฯลฯ ได้ในรูปแบบเดียวกัน)

# ==========================================
# เส้นที่ 2: Export File (ส่งคืนเป็นไฟล์ Excel)
# ==========================================
@app.post("/api/db/export", summary="ตรวจสอบ Config และ Export เป็น Excel")
def export_config(req: ConfigRequest):
    file_path = db_manager.check_vendor_config(
        vendor_id=req.vendor_id, 
        service_id=req.service_id, 
        action='export'
    )
    
    if not file_path or not os.path.exists(file_path):
        raise HTTPException(status_code=500, detail="ไม่สามารถสร้างไฟล์ Excel ได้")
        
    # คืนค่าเป็นไฟล์ให้ผู้ใช้ Download
    return FileResponse(path=file_path, filename=file_path, media_type='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')

# ==========================================
# เส้นที่ 3: Show Data (ส่งคืนเป็น JSON)
# ==========================================
@app.post("/api/db/show", summary="ตรวจสอบ Config และแสดงผลเป็น JSON")
def show_config(req: ConfigRequest):
    data = db_manager.check_vendor_config(
        vendor_id=req.vendor_id, 
        service_id=req.service_id, 
        action='show'
    )
    if not data:
        raise HTTPException(status_code=404, detail="ไม่พบข้อมูล Config")
    return {"status": "success", "data": data}

# ==========================================
# เส้นที่ 4: Get Transaction by TX_ID
# ==========================================
@app.post("/api/db/transaction", summary="ค้นหา Transaction จาก Database ด้วย TX_ID")
def get_transaction(req: TxIdRequest):
    df = db_manager.get_transaction_by_tx_id(tx_id=req.tx_id)
    
    if df.empty:
        raise HTTPException(status_code=404, detail=f"ไม่พบข้อมูล TX_ID: {req.tx_id} ในระบบ")
        
    # แปลง DataFrame เป็น JSON Dictionary
    # แทนที่ NaN ด้วย None เพื่อให้เข้ากันได้กับมาตรฐาน JSON
    result_json = df.where(pd.notnull(df), None).to_dict(orient='records')
    
    return {"status": "success", "tx_id": req.tx_id, "data": result_json}